In [1]:
using Plots, ProgressMeter, Statistics, BSON
include("analysis_tools.jl")

In [2]:
nx=100
ny=100
Lx=10.0
Ly=10.0
ν=0.1
kx=2
ky=2
u_mean=1.0
u_amplitude=0.5
v_amplitude=0.5
noise_strength=0.0005f0
t_end=15
cfl=0.8
nt=1000;

In [3]:
kxs = 1.0:0.5:3.0
kys = 1.0:0.5:3.0
u_amps = 0.1:0.3:2.0
v_amps = 0.1:0.3:2.0
means = [-1.0, 1.0]

kxs = [3]
kys = [2]
u_amps = [0.5]
v_amps = [0.5]
means = [1.0]

samples=200

data_length = samples*length(kxs)*length(kys)*length(u_amps)*length(v_amps)*length(means)

X = zeros(Float32, nx, ny, 2, data_length)
y = zeros(Float32, nx, ny, 2, data_length)

count=1
for kx in kxs, ky in kys, u_amplitude in u_amps, v_amplitude in v_amps, u_mean in means              
    X_data_u, X_data_v = burgers_FV_2D(nx, ny, Lx, Ly, ν, kx, ky, u_mean, u_amplitude, v_amplitude, noise_strength, t_end, cfl; nt)
    y_data_u, y_data_v = burgers_FV_2D(nx, ny, Lx, Ly, ν, kx, ky, u_mean, u_amplitude, v_amplitude, 0, t_end, cfl; nt)

    sample_idx = rand(1:length(X_data_u)-1, samples)
    for t in sample_idx
        X[:,:,1,count] .= X_data_u[t]
        X[:,:,2,count] .= X_data_v[t]

        y[:,:,1,count] .= y_data_u[t+1]
        y[:,:,2,count] .= y_data_v[t+1]
        count+=1
    end
    print("$(count)/$(data_length) $(round(100*count/data_length,digits=3))% done \r")
    flush(stdout)
end
Xμ, Xσ = mean(X), std(X)
# X = (X .- Xμ) ./ Xσ
# y = (y .- Xμ) ./ Xσ
println("Amount of data pairs: $(data_length)")

Amount of data pairs: 200


In [4]:
BSON.@save "data/burgers_datasetSmall2D.bson" X y Xμ Xσ